<a href="https://colab.research.google.com/github/Muralikrishna019/LLM-Fine-Tuning-for-Structured-Document-Extraction/blob/main/LLM_Fine_Tuning_for_Structured_Document_Extraction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load data and create a proper train/test split

In [1]:
import json
from sklearn.model_selection import train_test_split

data = json.load(open("/content/json_extraction_dataset_500.json", "r"))

train_data, test_data = train_test_split(data, test_size=0.15, random_state=42)
print(f"Train: {len(train_data)} | Test (held-out, never trained on): {len(test_data)}")

Train: 425 | Test (held-out, never trained on): 75


 Install Dependencies


In [2]:
!pip install unsloth trl peft accelerate bitsandbytes

Load base model + tokenizer

In [3]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
max_seq_length = 2048
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Shared prompt format + eval function


In [4]:
import re

def make_prompt(input_text):
    return f"### Input: {input_text}\n### Output:"

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

def evaluate(model, tokenizer, test_data, max_new_tokens=128):
    FastLanguageModel.for_inference(model)
    exact_matches = 0
    field_correct = 0
    field_total = 0
    results = []

    for ex in test_data:
        prompt = make_prompt(ex["input"])
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # deterministic for fair comparison
            use_cache=True,
        )
        decoded = tokenizer.decode(output[0], skip_special_tokens=True)
        completion = decoded[len(prompt):]
        pred_json = extract_json(completion)
        gt_json = ex["output"]

        is_exact = (pred_json == gt_json)
        exact_matches += int(is_exact)

        if pred_json:
            for k, v in gt_json.items():
                field_total += 1
                if pred_json.get(k) == v:
                    field_correct += 1
        else:
            field_total += len(gt_json)

        results.append({
            "input": ex["input"], "gt": gt_json,
            "pred": pred_json, "exact": is_exact
        })

    exact_match_acc = exact_matches / len(test_data) * 100
    field_acc = (field_correct / field_total * 100) if field_total else 0
    return exact_match_acc, field_acc, results

Baseline eval

In [5]:
baseline_exact, baseline_field, baseline_results = evaluate(model, tokenizer, test_data)
print(f"BASELINE   — Exact Match: {baseline_exact:.1f}% | Field Accuracy: {baseline_field:.1f}%")

Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

BASELINE   — Exact Match: 0.0% | Field Accuracy: 0.0%


Add LoRA adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.7.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Format TRAIN split only

In [7]:
from datasets import Dataset

def format_prompt(example):
    return f"### Input: {example['input']}\n### Output: {json.dumps(example['output'])}<|endoftext|>"

formatted_train = [format_prompt(item) for item in train_data]
dataset = Dataset.from_dict({"text": formatted_train})

Train

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=0, # Changed from 2 to 0 to avoid PicklingError
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="no", # Changed from "epoch" to "no" to avoid PicklingError during saving
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/425 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 425 | Num Epochs = 3 | Total steps = 162
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)


Step,Training Loss
25,0.123089
50,0.120817
75,0.113896
100,0.112885
125,0.107769
150,0.107168


Eval AFTER fine-tuning, on the SAME held-out test set

In [11]:
finetuned_exact, finetuned_field, finetuned_results = evaluate(model, tokenizer, test_data)
print(f"FINE-TUNED — Exact Match: {finetuned_exact:.1f}% | Field Accuracy: {finetuned_field:.1f}%")

print("\n=== SUMMARY ===")
print(f"{'Metric':<20}{'Baseline':<12}{'Fine-tuned':<12}{'Delta':<10}")
print(f"{'Exact Match %':<20}{baseline_exact:<12.1f}{finetuned_exact:<12.1f}{finetuned_exact-baseline_exact:+.1f}")
print(f"{'Field Accuracy %':<20}{baseline_field:<12.1f}{finetuned_field:<12.1f}{finetuned_field-baseline_field:+.1f}")

Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=12

FINE-TUNED — Exact Match: 100.0% | Field Accuracy: 100.0%

=== SUMMARY ===
Metric              Baseline    Fine-tuned  Delta     
Exact Match %       0.0         100.0       +100.0
Field Accuracy %    0.0         100.0       +100.0


In [12]:
failures = [r for r in finetuned_results if not r["exact"]][:5]
for f in failures:
    print("INPUT:", f["input"][:120])
    print("EXPECTED:", f["gt"])
    print("PREDICTED:", f["pred"])
    print("---")

Save/export

In [13]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in gguf_model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gguf_model.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors: reconstructing file:   0%|          |  0.00B / 2.65GB            

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:36<00:00, 78.02s/it]


Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:40<01:40, 100.92s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:29<00:00, 74.52s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10079-mix-fb3d4ca (app-b10079-mix-fb3d4ca-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_model_gguf/phi-3-mini-4k-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed success

{'save_directory': 'gguf_model',
 'gguf_directory': 'gguf_model_gguf',
 'gguf_files': ['gguf_model_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'gguf_model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}